<table>
  <tr>
    <td><div align="left"><font size="30">Spatial Math for Robotics & Vision</font></div></td>
    <td><img src="support/figs/CartesianSnakes_LogoW.png" width="300"></td>
  </tr>
</table>

<p></p>
<div align="center" style="font-size: 1.5em;">🏗️ Foundations</div>
<p></p>


(c) Peter Corke 2026

In [ ]:
import importlib
import setup_tutorial
importlib.reload(setup_tutorial)
await setup_tutorial.setup_tutorial(required_packages=['matplotlib', 'numpy', 'scipy'])

print("\nspatialmath version:", importlib.import_module("spatialmath").__version__)


In [ ]:
# import matplotlib.pyplot as plt
import numpy as np
np.set_printoptions(linewidth=100, formatter={'float': lambda x: f"{x:8.3g}" if abs(x) > 1e-10 else f"{0:8.3g}"})

# Spatial Math

In robotics and computer vision we frequently represent pose in 3D space as a matrix or a vector.  NumPy is great for matrices, but in robotics and computer vision we need matrices and vectors that have **special** structure.

* All $\mathrm{SE(3)}$ poses can be represented by a $4 \times 4$ matrix.
* Not all $4 \times 4$ matrices are $\mathrm{SE(3)}$

* Pose in 3D space can be represented by a subset of all possible 4x4 matrices:
  * the **special** Euclidean group of order 3.
  * the tangent space $\mathrm{se(3)}$ which has 6 unique elements we can represent in a **special** 6-vector
* Orientation in 3D space be represented by a unit quaternion, a 4-vector with one constraint
* Lines in 3D can be represented by a 6-vector with 2 contraints (homogeneous vector and the Klein quadric)

The `SE3` class is a gatekeeper class.  It contains a NumPy array of shape `(4,4)` but it only admits matrices that belong to the group $\mathrm{SE(3)}$


Let's start simply

In [ ]:
from spatialmath import SE3

SE3()

which is the identity matrix.

We could write this more laboriously as

In [ ]:
SE3(np.eye(4))

The matrix was admitted by the gatekeeper because it belongs to $\mathrm{SE(3)}$.

Here's an example of the gatekeeper at work, keeping out the undeserving

In [ ]:
try:
    SE3(np.zeros((4, 4)))
except ValueError as e:
    print("bad thing")

There are various class methods that act like constructors.  Here's a very complex way to compound motion in the X-, Y-, and Z-directions.

In [ ]:
SE3.Tx(1) * SE3.Ty(2) * SE3.Tz(3)

Note that `*` is the group operator and means composition.  In Python terms it's matrix multiplication, `@` operator inside the box.

Now a sequence of translations and rotations about the canonical axes

In [ ]:
T = SE3.Tx(1) * SE3.Rx(np.pi/4) * SE3.Tz(2) * SE3.Ry(np.pi/5) * SE3.Rz(np.pi/6)
print(T)

To understand the position and orientation of this pose it's very helpful to plot it

In [ ]:
T.plot()

In [ ]:
T.printline()

The upper left $3 \times 3$ matrix belongs to the group $\mathrm{SO(3)}$ and that has its own class

In [ ]:
from spatialmath import SO3

R = SO3(T)
R

and we can convert it to various other representations.  Roll/pitch/yaw angles are common, but it's really important to specify the order of rotation. In this example it's: Rx(yaw) Ry(pitch) Rz(roll)

In [ ]:
R.rpy(order="xyz")

or in angle-axis form: a rotation about an axis specified as a unit vector

In [ ]:
R.angvec()

or as a unit-quaternion (specifically a right-handed quaternion, not a left-handed or JPL quaternion)

In [ ]:
from spatialmath import UnitQuaternion

q = UnitQuaternion(R)
q

The `*` operator also performs composition of `SO3` and `UnitQuaternion` objects

In [ ]:
UnitQuaternion(R*R)

In [ ]:
q*q

and we see the equivalence of rotational composition using rotation matrices or unit quaternions.

The rotational conversions above have equivalent methods for `SE3` object, they just ignore the translation component

In [ ]:
print(T.rpy())
print(UnitQuaternion(T))

All the objects are subclasses of `list` and so can do all the things you can do with a list: slice, iterate, insert, append, pop etc.

In [ ]:
R = SO3([SO3.Rx(np.pi/2), SO3.Ry(np.pi/3), SO3.Rz(np.pi/4)])
print(len(R))
print(R[1])

for _ in R:
    print(_.rpy())


Let's create a longer list, rotations around the Y-axis from 0 to $\pi/2$ radians

In [ ]:
Rt = SO3.Ry([theta for theta in np.linspace(0, np.pi/2, 50)])
print(len(Rt))
print(Rt[0])
print(Rt[49])

In one line we can convert that to a listy quaternion

In [ ]:
qt = UnitQuaternion(Rt)
print(len(qt))
print(qt[0])
print(qt[49])

We can also do broadcasting very easily.  Here we apply $R_x(\pi/2)$ to every element of `Rt`

In [ ]:
Rtx = SO3.Rx(np.pi/2) * Rt
print(len(Rtx))
print(Rtx[0])
print(Rtx[49])

Let's return to the somewhat random $\mathrm{SE(3)}$ matrix we computed earlier

In [ ]:
T

and compute its logarithm

In [ ]:
T.log()

which is another 4x4 matrix with special (different special) structure.  The bottom row is zero, and the upper left $3 \times 3$ matrix is skew symmetric.  It has only 6 unique values.

We can represent this as a twist $(v, \omega)$, where $v \in \mathbb{R}^3$ is the moment and encodes the position of the screw
axis in space and the pitch of the screw, and $\omega \in \mathbb{R}^3$ is the direction of the screw axis.

In [ ]:
from spatialmath import Twist3

S = Twist3(T)
print(S)

which contains all the information needed to reconstruct the original $\mathrm{SE(3)}$ matrix

In [ ]:
S.SE3()

The twist encodes motion as a rotation of a frame around a screw axis.  For this example, the pitch of the screw is 

In [ ]:
h = S.pitch
print(h)

that is, for every radian of rotation about the axis, we translate `h` along the axis.

The axis of the screw is

In [ ]:
S.line()

which is of type

In [ ]:
type(_)

which is a Plücker representation of a 3D line, another 6-vector, but this one has 2 contraints.

The pose represented by `T` can be obtained by rotating a frame from the origin around a screw, with the pitch and axis described above, by an angle of 

In [ ]:
print(S.theta, "radians")

# and there's lots more

We've barely scratched the surface. There are also:

* lots more conversions back and forth between the representations we've touched on.
* 2D versions of all of this: `SE2`, `SO2` and `Twist2`
* classes for 3D planes and lines (which we touched on), and methods for point-line distance, line-line and line-plane intersection.
* a handy graphics library for lines, arrows, boxe, circle, ellipses, etc.

